# 07 — VideoMAE V2 RGB-only baseline (demo ASL Citizen top-20)

Đây là **baseline nghiên cứu RGB-only**, chưa phải thí nghiệm cuối trên 200 lớp. Notebook lấy đúng 20 hạng đầu từ báo cáo ASL-LEX top-200, giữ nguyên toàn bộ membership của ba split chính thức `train` / `validation` / `test` và kiểm tra không trùng `sample_id`. Không dùng pose, graph encoder hay feature fusion.

Mặc định backbone VideoMAE V2 được đóng băng và chỉ học classifier 20 lớp. Giới hạn epoch/batch giúp xem nhanh output; mọi metric phải ghi rõ là **demo 20 lớp, đánh giá hữu hạn**. Khi nghiên cứu chính thức phải chạy lại 200 lớp, full schedule và chỉ mở test sau khi đã chốt mô hình bằng validation. Checkpoint nền là [OpenGVLab/VideoMAEv2-Base](https://huggingface.co/OpenGVLab/VideoMAEv2-Base), giấy phép CC BY-NC 4.0.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/asl-citizen-videomaev2-demo-baseline'  # @param {type:'string'}
RESULTS_ROOT_STR = '/content/drive/.shortcut-targets-by-id/1-oYEcvJh4ylv_f4AKkJkCjs3FgzjDBFE/silent-signal-results/asl_citizen'  # @param {type:'string'}
DATASET_ROOT_STR = '/content/ASL_Citizen'  # @param {type:'string'}
STREAM_EXTRACT_SELECTED = True  # @param {type:'boolean'}
ACCEPT_ASL_CITIZEN_LICENSE = False  # @param {type:'boolean'}
MODEL_ID = 'OpenGVLab/VideoMAEv2-Base'  # @param {type:'string'}
MODEL_REVISION = '0e826d7e85e39f9d951e331cd91c5c2d8142d385'  # @param {type:'string'}
CLASS_COUNT = 20  # @param {type:'integer'}
MAX_EPOCHS = 3  # @param {type:'integer'}
BATCH_SIZE = 2  # @param {type:'integer'}
MAX_TRAIN_BATCHES = 60  # @param {type:'integer'}
MAX_EVAL_BATCHES = 40  # @param {type:'integer'}
CHECKPOINT_EVERY = 20  # @param {type:'integer'}
PROGRESS_EVERY = 5  # @param {type:'integer'}
LEARNING_RATE = 0.0003  # @param {type:'number'}
NUM_WORKERS = 2  # @param {type:'integer'}
DEVICE = 'auto'  # @param ['auto', 'cuda', 'cpu']
FREEZE_BACKBONE = True  # @param {type:'boolean'}
RESUME = True  # @param {type:'boolean'}
RUN_TEST = False  # @param {type:'boolean'}
RUN_TRAINING = True  # @param {type:'boolean'}

if CLASS_COUNT != 20:
    raise ValueError('Notebook demo này được cố định ở đúng 20 từ.')
PROJECT_ROOT = Path('/content/silent-signal')
RESULTS_ROOT = Path(RESULTS_ROOT_STR)
DATASET_ROOT = Path(DATASET_ROOT_STR)
TOP200_ROOT = RESULTS_ROOT / 'subsets/asl_citizen_asllex_top200'
SOURCE_MANIFEST = TOP200_ROOT / 'manifest.csv'
SOURCE_SELECTION = TOP200_ROOT / 'selection_report.json'
BASELINE_ROOT = TOP200_ROOT / 'baselines/videomaev2_rgb_demo20'

## Sơ đồ baseline và ranh giới thí nghiệm

```text
ASL Citizen video ──► 16 RGB frames ──► VideoMAE V2 Base ──► Linear head 20 lớp
        │                                           │
        └── official train/validation/test          └── không có pose/graph/fusion
```

`validation` dùng chọn checkpoint và phân tích lỗi. `test` mặc định chưa chạy; không được dùng test để chỉnh kiến trúc.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig, ax = plt.subplots(figsize=(13, 3))
boxes = [('Official RGB video', 0.02, '#457b9d'), ('16 frames\n224×224', 0.27, '#2a9d8f'), ('VideoMAE V2 Base', 0.52, '#e9c46a'), ('Classifier\n20 words', 0.77, '#e76f51')]
for label, x, color in boxes:
    ax.add_patch(FancyBboxPatch((x, 0.35), 0.19, 0.32, boxstyle='round,pad=0.03', facecolor=color, alpha=0.9))
    ax.text(x + 0.095, 0.51, label, ha='center', va='center', color='white', fontsize=11, weight='bold')
for x in (0.21, 0.46, 0.71):
    ax.add_patch(FancyArrowPatch((x, 0.51), (x + 0.055, 0.51), arrowstyle='-|>', mutation_scale=18))
ax.text(0.5, 0.12, 'DEMO: capped epochs/batches — final study phải chạy lại đủ 200 lớp', ha='center', color='#9d0208', weight='bold')
ax.set(xlim=(0, 1), ylim=(0, 1)); ax.axis('off'); plt.show()

## Lấy đúng nhánh và cài môi trường Colab

In [ ]:
import shutil, subprocess, sys, time
def run(command):
    command = list(map(str, command))
    started = time.perf_counter()
    print(time.strftime('[%H:%M:%S] START'), ' '.join(command), flush=True)
    subprocess.run(command, check=True)
    print(time.strftime('[%H:%M:%S] DONE '), f'{time.perf_counter() - started:.1f}s', flush=True)

if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch', PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only', 'origin', PROJECT_GIT_REF])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', PROJECT_ROOT])
run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.48.3', 'timm==1.0.15', 'easydict==1.13', 'opencv-python-headless==4.10.0.84', 'matplotlib==3.10.0'])
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Chốt 20 từ và kiểm tra split trước khi tải video

20 từ là **20 hạng đầu của `SignFrequency(M)` trong selection report top-200**, không phải 20 lớp có nhiều video nhất. Notebook chỉ lọc class; membership split chính thức không đổi.

In [ ]:
import csv, json
from collections import Counter
if not SOURCE_MANIFEST.is_file(): raise FileNotFoundError(SOURCE_MANIFEST)
if not SOURCE_SELECTION.is_file(): raise FileNotFoundError(SOURCE_SELECTION)
selection = json.loads(SOURCE_SELECTION.read_text(encoding='utf-8'))
classes = selection['classes'][:CLASS_COUNT]
if [item['rank'] for item in classes] != list(range(1, CLASS_COUNT + 1)):
    raise RuntimeError('20 lớp không phải 20 rank ASL-LEX đầu tiên.')
with SOURCE_MANIFEST.open(encoding='utf-8-sig', newline='') as handle:
    all_rows = list(csv.DictReader(handle))
selected_indices = {str(item['subset_class_index']) for item in classes}
rows = [row for row in all_rows if row['class_index'] in selected_indices]
ids = {split: {row['sample_id'] for row in rows if row['split'] == split} for split in ('train', 'validation', 'test')}
if ids['train'] & ids['validation'] or ids['train'] & ids['test'] or ids['validation'] & ids['test']:
    raise RuntimeError('LEAKAGE: sample_id xuất hiện ở nhiều split.')
if sum(map(len, ids.values())) != len(rows): raise RuntimeError('Split thiếu hoặc sample_id trùng.')
counts = Counter((int(row['class_index']), row['split']) for row in rows)
print(f"{'rank':>4}  {'gloss':<25} {'train':>13} {'validation':>13} {'test':>13}")
for item in classes:
    index = item['subset_class_index']; total = sum(counts[index, split] for split in ids)
    values = [f"{counts[index, split]} ({100 * counts[index, split] / total:.1f}%)" for split in ('train', 'validation', 'test')]
    if any(counts[index, split] == 0 for split in ids): raise RuntimeError(f"{item['gloss_name']} thiếu một official split.")
    print(f"{item['rank']:>4}  {item['gloss_name'][:25]:<25} {values[0]:>13} {values[1]:>13} {values[2]:>13}")
print('\nSplit isolation: PASS | clips:', {split: len(value) for split, value in ids.items()})

## Stream-extract đúng video của 20 từ

Cell này đọc trực tiếp các member cần thiết trong ZIP chính thức của Microsoft và không lưu ZIP. Nó không extract toàn bộ dataset 43 GB. File đã hoàn tất được CRC-check và dùng lại nếu chạy lại trong cùng runtime.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
selected_video_paths = {row['video_path'] for row in rows}
missing = sorted(path for path in selected_video_paths if not (DATASET_ROOT / path).is_file())
print(f'Video hiện có: {len(selected_video_paths) - len(missing)}/{len(selected_video_paths)}; thiếu: {len(missing)}', flush=True)
if missing:
    if not STREAM_EXTRACT_SELECTED:
        raise FileNotFoundError('Thiếu video và STREAM_EXTRACT_SELECTED=False.')
    if not ACCEPT_ASL_CITIZEN_LICENSE:
        raise RuntimeError('Đọc điều khoản Microsoft rồi bật ACCEPT_ASL_CITIZEN_LICENSE=True.')
    from silent_signal.data.asl_download import extract_remote_archive
    extract_remote_archive(DATASET_ROOT, include_paths=selected_video_paths)
remaining = [path for path in selected_video_paths if not (DATASET_ROOT / path).is_file()]
if remaining: raise FileNotFoundError(f'Vẫn thiếu {len(remaining)} video sau extract.')
print(f'Dataset demo sẵn sàng: {len(selected_video_paths)}/{len(selected_video_paths)} video', flush=True)

## Train có log, giới hạn và resume

Log hiện `batch hiện tại/tổng`, sample, loss, top-1 và ETA. `last_checkpoint.pt` được ghi mỗi `CHECKPOINT_EVERY` batch lên Drive. Nếu Colab ngắt, mở lại notebook, giữ nguyên cấu hình và `RESUME=True`. Tăng `MAX_EPOCHS` nếu muốn chạy tiếp thêm epoch.

In [ ]:
def run_stream(command):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout: print(line, end='', flush=True)
    code = process.wait()
    if code: raise subprocess.CalledProcessError(code, command)

command = [sys.executable, '-u', '-m', 'silent_signal.cli.train_videomaev2_demo',
    '--manifest', SOURCE_MANIFEST, '--selection-report', SOURCE_SELECTION,
    '--dataset-root', DATASET_ROOT, '--output-root', BASELINE_ROOT,
    '--model-id', MODEL_ID, '--model-revision', MODEL_REVISION,
    '--classes', CLASS_COUNT, '--epochs', MAX_EPOCHS, '--batch-size', BATCH_SIZE,
    '--learning-rate', LEARNING_RATE, '--max-train-batches', MAX_TRAIN_BATCHES,
    '--max-eval-batches', MAX_EVAL_BATCHES, '--checkpoint-every', CHECKPOINT_EVERY,
    '--progress-every', PROGRESS_EVERY, '--num-workers', NUM_WORKERS, '--device', DEVICE]
if FREEZE_BACKBONE: command.append('--freeze-backbone')
if RESUME: command.append('--resume')
if RUN_TEST: command.append('--run-test')
if RUN_TRAINING: run_stream(command)
else: print('RUN_TRAINING=False — chỉ kiểm tra dữ liệu, chưa train.')

## Kết quả bền vững trên Drive

In [ ]:
from IPython.display import Image, display
report_path = BASELINE_ROOT / 'baseline_report.json'
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps(report, ensure_ascii=False, indent=2))
    curve = BASELINE_ROOT / 'training_curves.png'
    if curve.is_file(): display(Image(filename=str(curve)))
    print('PASS demo baseline. Tiếp theo chạy notebook 08 trên validation_predictions.csv.')
else:
    print('Chưa có baseline_report.json. Nếu vừa dừng giữa chừng, checkpoint vẫn ở:', BASELINE_ROOT / 'last_checkpoint.pt')